In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

# ========= 1) 从 Excel 读取指定列 =========
file_path = r"your_data.xlsx"         # Excel 路径(str)
sheet_name = "Sheet1"                 # 工作表名(str)
time_col = "time"                     # 时间列标题(str)
target_col = "y"                      # 目标列标题(str)

df = pd.read_excel(file_path, sheet_name=sheet_name)
t_train = df[time_col].to_numpy(dtype=float)   # 训练时间序列(np.ndarray)
y_train = df[target_col].to_numpy(dtype=float) # 训练目标值(np.ndarray)

# ========= 2) 参数模板（你要填） =========
params = {
    "K_init": 100.0,       # float: Logistic 上限初值
    "r_init": 0.1,         # float: 增长率初值
    "t0_init": 10.0,       # float: 拐点初值
    "t_future": np.array([21,22,23], dtype=float)  # np.ndarray: 未来时间点
}

def logistic_func(t, K, r, t0):
    """Logistic 曲线函数"""
    return K / (1 + np.exp(-r * (t - t0)))

popt, _ = curve_fit(
    logistic_func, t_train, y_train,
    p0=[params["K_init"], params["r_init"], params["t0_init"]],
    maxfev=10000
)
y_pred = logistic_func(params["t_future"], *popt)
print("拟合参数(K,r,t0)=", popt)
print("预测值=", y_pred)


In [ ]:
"""
Logistic 预测模型

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "Logistic 预测模型.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
TARGET_COLUMN = "y"  # TODO: 请填写[目标列名]，说明：应为逐渐接近上限的增长序列。
K_INIT = 100.0  # TODO: 请填写[容量上限初值]，说明：应大于历史最大值。
A_INIT = 10.0  # TODO: 请填写[形状参数初值]，说明：正数。
B_INIT = 0.2  # TODO: 请填写[增长率初值]，说明：正数。
FORECAST_STEPS = 5  # TODO: 请填写[预测期数]，说明：不宜过长。



REQUIRES_DATA = True  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def logistic(t, K, a, b):
    return K / (1 + a * np.exp(-b * t))


def run_model(data: pd.DataFrame) -> None:
    t = np.arange(len(data), dtype=float)
    y = data[TARGET_COLUMN].to_numpy(dtype=float)
    params, _ = curve_fit(logistic, t, y, p0=[K_INIT, A_INIT, B_INIT], maxfev=10000)
    future_t = np.arange(len(data) + FORECAST_STEPS, dtype=float)
    fitted_forecast = logistic(future_t, *params)
    result = pd.DataFrame({"序号": future_t + 1, "拟合或预测值": fitted_forecast})
    result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print("参数 K, a, b:", params)
    print(result.tail())


if __name__ == "__main__":
    df = load_data()
    run_model(df)
